In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Local-dev fallback: in production, excel-creator's file:// dependency in
# pyproject.toml makes sheet_experiment/experiment_excel_builder importable
# once pip-installed. For local notebook kernels where that hasn't happened,
# locate apps/Excel_creator (a sibling of this notebook's own directory) by
# trying several signals, since which one actually works depends on the
# kernel/IDE: VS Code's own notebook-file global, the process's working
# directory, and any sys.path entry that already points at this app.
_candidate_dirs = []
_vsc_path = globals().get("__vsc_ipynb_file__")
if _vsc_path:
    _candidate_dirs.append(("__vsc_ipynb_file__", Path(_vsc_path).resolve().parent))
_candidate_dirs.append(("cwd", Path.cwd()))
_candidate_dirs.extend(
    ("sys.path", Path(p)) for p in sys.path if Path(p).name == "smart_databaser"
)

_EXCEL_CREATOR_DIR = None
for _source, _candidate_dir in _candidate_dirs:
    _guess = _candidate_dir.parent / "Excel_creator"
    if (_guess / "experiment_excel_builder.py").exists():
        _EXCEL_CREATOR_DIR = _guess
        print(f"Found apps/Excel_creator via {_source}: {_guess}")
        break

if _EXCEL_CREATOR_DIR is None:
    print("DEBUG - could not locate apps/Excel_creator automatically. Tried:")
    for _source, _candidate_dir in _candidate_dirs:
        print(f"  {_source}: {_candidate_dir} -> {_candidate_dir.parent / 'Excel_creator'}")
    print("DEBUG - full sys.path:")
    for _p in sys.path:
        print("   ", repr(_p))
    raise RuntimeError(
        "Could not find apps/Excel_creator from this notebook. Set "
        "_EXCEL_CREATOR_DIR manually (e.g. "
        "_EXCEL_CREATOR_DIR = Path(r'C:\\path\\to\\nomad_voila\\apps\\Excel_creator')) "
        "using the DEBUG output above, then re-run."
    )

if str(_EXCEL_CREATOR_DIR) not in sys.path:
    sys.path.insert(0, str(_EXCEL_CREATOR_DIR))

from app import initialize_ui
from IPython.display import HTML, display
from IPython.display import clear_output as _clear_cell

from hysprint_utils.access_token import get_token, log_notebook_usage
from hysprint_utils.config import API_ENDPOINT, URL_BASE

URL_API = URL_BASE + API_ENDPOINT

# Clear any saved or previously accumulated cell outputs.
# VS Code Jupyter does not clear cell outputs on re-run; without this, each run
# (and each notebook open) adds a new copy of all widgets below the old ones.
# wait=True means the clear fires on the first new output below, avoiding a
# blank-screen flash.
_clear_cell(wait=True)
display(HTML("<script>document.title='Smart Databaser'</script>"))
log_notebook_usage()

TOKEN = get_token(URL_API)

print("Initialising UI...")
try:
    display(initialize_ui(URL_API, TOKEN))
except Exception as _init_err:
    import traceback as _tb
    print(f"UI initialisation failed: {_init_err}")
    _tb.print_exc()
